In [ ]:
import pandas as pd 
import numpy as np
import re
import json

In [ ]:
with open('../card_parser/smartphones_results_over.json', 'r', encoding='utf-8') as f:
    data_smartphone = json.load(f) 
df_smartphone = pd.json_normalize(data_smartphone)

with open('../card_parser/tablets_results_over.json', 'r', encoding='utf-8') as f:
    data_tablet = json.load(f)
df_tablet = pd.json_normalize(data_tablet)

In [ ]:


cols_to_fix = ['Price', 'Old Price', 'Installment']

df_smartphone[cols_to_fix] = df_smartphone[cols_to_fix].apply(lambda x: x.str.replace(r'\D', '', regex=True))

df_tablet[cols_to_fix] = df_tablet[cols_to_fix].apply(lambda x: x.str.replace(r'\D', '', regex=True))



In [ ]:


print(df_tablet.shape)
mask = df_tablet.iloc[: , -101:-1].notna().any(axis = 1)
df_tablet_after_dropping = df_tablet[~mask].iloc[:, :-101]
print(df_tablet_after_dropping.shape)



In [ ]:


handle_colss = ['Characteristics.Оперативная память','Characteristics.Тип',
'Characteristics.Частота обновления экрана, Гц',
'Characteristics.Частота процессора, ГГц']

tablet_colms_drop_list = ['Characteristics.Обязательные программы предустановлены',
 'Characteristics.Работа в режиме ожидания, ч',
 'Characteristics.Беспроводные интерфейсы',
 'Characteristics.Стандарты связи',
 'Characteristics.Форм-фактор SIM',
 'Characteristics.Модуль связи Bluetooth',
 'Characteristics.Модуль связи WiFi',
 'Characteristics.Навигация',
 'Characteristics.Разъемы',
 'Characteristics.Назначение слотов',
 'Characteristics.Работа в режиме телефона',
 'Characteristics.Встроенные датчики',
 'Characteristics.Функции камеры',
 'Characteristics.Особенности',
 'Characteristics.Макс. время работы (музыка),  ч',
 'Characteristics.Макс. время работы (видео),  ч',
 'Characteristics.Работа в режиме ожидания, ч',
 'Characteristics.Тип карты памяти',
 'Characteristics.Макс. объем карты памяти,  ГБ'
 ]

tablet_transformation_dict = {'Category':['category','object'],
 'Name':['name_card','object'],
 'Price':['price','numeric'],
 'Old Price':['price_old','numeric'],
 'Installment':['price_installment','numeric'],
 'Rating':['rating_card','float'],
 'Reviews':['reviews_card','int'],
 'Seller':['name_seller','object'],
 'Seller Rating':['rating_seller','float'],
 'URL':['url','object'],
 'Characteristics.Артикул':['article', 'int'],
 'Characteristics.Диагональ экрана, дюймы':['screen_diagonal', 'float'],
 'Characteristics.Оперативная память':['ram','float'],
 'Characteristics.Операционная система':['os','object'], #CATEGORY
 'Characteristics.Встроенная память,  ГБ':['internal_storage','int'],
 'Characteristics.Разрешение экрана':['screen_size','object'],
 'Characteristics.Процессор': ['CPU', 'object'],
 'Characteristics.Разрешение основной камеры, Мпикс':['main_camera','float'],
 'Characteristics.Технология матрицы':['matrix','object'],
 'Characteristics.Тип':['type_of_tablet','object'], #CATEGORY
 'Characteristics.Бренд':['brand','object'], #CATEGORY
 'Characteristics.Цвет':['color','object'], 
 'Characteristics.Год анонсирования':['announcement_year','int'], #CATEGORY
 'Characteristics.Версия Android':['android_version','object'], #CATEGORY
 'Characteristics.Версия iOS':['ios_version', 'object'], #CATEGORY
 'Characteristics.Версия HarmonyOS': ['harmonyos_version','object'], #CATEGORY
 'Characteristics.Частота обновления экрана, Гц':['hz_screen','int'],
 'Characteristics.Плотность пикселей, ppi':['ppi','float'],
 'Characteristics.Бренд процессора':['processor_brand','object'], #CATEGORY
 'Characteristics.Бренд графического процессора':['brand_graphic_proccessor','object'], #CATEGORY
 'Characteristics.Число ядер процессора':['processor_cores','int'],
 'Characteristics.Частота процессора, ГГц':['ghz_processor','float'],
 'Characteristics.Видеопроцессор':['video_processor','object'],
 'Characteristics.Модуль сотовой связи':['cellular_network_module', 'object'],
 'Characteristics.Число физических SIM-карт':['count_sim','object'], 
 'Characteristics.Слот для карты памяти':['memory_card_slot','object'],
 'Characteristics.Разрешение фронтальной (селфи) камеры, Мпикс':['front_camera','float'],
 'Characteristics.Наличие клавиатуры':['keyboard','object'],
 'Characteristics.Основной материал корпуса':['case_material','object'],
 'Characteristics.Защита от влаги':['waterproof','object'],
 'Characteristics.Емкость аккумулятора, мАч':['battery_capacity','int'],
 'Characteristics.Аксессуары в комплекте':['accessories_included','object'],
 'Characteristics.Размеры, мм':['dimensions','object'],
 'Characteristics.Вес товара, г':['weight_g','float'],
 'Characteristics.Гарантийный срок':['warranty_period','object'],
 'Characteristics.Страна-изготовитель':['country_of_origin','object'], #CATEGORY
 'Characteristics.Срок службы, лет':['service_life','float'],
}



In [ ]:
def convert_to_gb(df , work_col):
    s = df[work_col].astype(str).str.replace('Отсутствует', '', case=False).str.replace(',', '.')
    
    numbers = s.str.extract(r'(\d+\.?\d*)', expand=False).astype(float)
    
    is_mb = s.str.contains('МБ|MB', case=False, na=False)
    is_tb = s.str.contains('ТБ|TB', case=False, na=False)
    
    numbers = np.where(is_mb, numbers / 1024, numbers)
    numbers = np.where(is_tb, numbers * 1024, numbers)
    
    return numbers


def convert_hz_to_num(df,work_col):
 numbers = df[work_col].str.extract(r'(\d+\.?\d*)', expand=False)
 return numbers

def transform_dataframe(df, transformation_dict):
    df = df.copy()

    rename_map = {k: v[0] for k, v in transformation_dict.items()}
    df = df.rename(columns=rename_map)
    
    for old_name, info in transformation_dict.items():
        new_name = info[0]
        target_type = info[1]
        
        if new_name in df.columns:
            try:
                if target_type in ['numeric', 'float']:
                    df[new_name] = pd.to_numeric(df[new_name], errors='coerce')
                elif target_type == 'int':
                    df[new_name] = pd.to_numeric(df[new_name], errors='coerce').astype('Int64')
                elif target_type == 'object':
                    df[new_name] = df[new_name].astype('object') 
            except Exception as e:
                print(f"⚠️ Failed to convert '{new_name}': {e}")
    return df
    

In [ ]:


df_tablet_after_dropping[handle_colss[0]] = convert_to_gb(df_tablet_after_dropping, handle_colss[0])
df_tablet_after_dropping[handle_colss[1]] = df_tablet_after_dropping[handle_colss[1]].astype(str).replace('Воротник для животных', np.nan)
df_tablet_after_dropping[handle_colss[2]] = convert_hz_to_num(df_tablet_after_dropping, handle_colss[2])
df_tablet_after_dropping[handle_colss[3]] = pd.to_numeric(
    df_tablet_after_dropping[handle_colss[3]]
    .astype(str)
    .str.replace(',', '.')
    .str.extract(r'(\d+\.?\d*)', expand=False), 
    errors='coerce'
)



In [ ]:
df_tablet = df_tablet_after_dropping.drop(tablet_colms_drop_list, axis=1)
df_tablet = transform_dataframe(df_tablet, tablet_transformation_dict)

In [ ]:
# print(df_tablet.columns)
print(df_tablet.info())
# df_tablet.sample(5)

In [ ]:
print(df_smartphone.shape)
mask = df_smartphone.iloc[: , -150:-1].notna().any(axis = 1)
df_smartphone_after_dropping = df_smartphone[~mask].iloc[:, :-150]
print(df_smartphone_after_dropping.shape)

In [ ]:


for column in df_smartphone_after_dropping.columns:
    print(f"Колонка: {column}")
    print(df_smartphone_after_dropping[column].unique())
    print("-" * 30) 



In [ ]:
# Убираем ограничение на количество выводимых строк
pd.set_option('display.max_rows', None)# Теперь ваш вывод покажет всё до конца
# Убираем ограничение на количество колонок
pd.set_option('display.max_columns', None)
# Увеличиваем ширину вывода, чтобы текст не обрезался
pd.set_option('display.expand_frame_repr', False)


# pd.reset_option('all')

In [ ]:
df_table.shape